# Diffusion tensor imaging: mapping nerve fibres

*medical · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/medical/diffusion_tensor_imaging)

Water diffuses faster along axons than across them, so each voxel has a diffusion tensor $D_{ij}$. The Stejskal–Tanner signal $S = S_0\,e^{-b\,\hat g_i D_{ij}\hat g_j}$ is one einsum over voxels and gradient directions; a batched least-squares fit of the log-signals recovers $D_{ij}$, its eigen-decomposition gives mean diffusivity, fractional anisotropy and the fibre direction, and following that direction with RK2 traces the nerve bundles (tractography).

**Vector form**

$$S(\hat{\mathbf g}) = S_0\,\exp\!\bigl(-b\,\hat{\mathbf g}^{\mathsf T}\mathbf D\,\hat{\mathbf g}\bigr),\qquad \mathbf D = \textstyle\sum_a \lambda_a\,\mathbf e_a\mathbf e_a^{\mathsf T},\qquad \dot{\mathbf x} = \mathbf e_1(\mathbf x)$$

**Index (tensor) form**

$$S_{vg} = S_{0v}\,e^{-b\,\hat g_{gi}\,D_{vij}\,\hat g_{gj}},\qquad \ln\frac{S_{vg}}{S_{0v}} = -b\,B_{gm}\,d_{vm},\qquad MD_v = \tfrac13 D_{vii},\qquad FA_v = \sqrt{\tfrac32}\,\frac{\lVert\lambda_{va} - MD_v\rVert}{\lVert\lambda_{va}\rVert}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\hat g_{gi}$ | `gradient_directions(G)` | (G, 3) | unit diffusion-gradient directions spread over a hemisphere |
| $D_{vij}$ | `phantom(n)` | (V, 3, 3) | true diffusion tensor of every voxel: bundles $\lambda_\parallel e_ie_j + \lambda_\perp(\delta_{ij} - e_ie_j)$, isotropic CSF and grey matter |
| $S_{vg}$ | `S0 * exp(-b * einsum('gi,vij,gj->vg', g, D, g))` | (V, G) | repeated $i, j$ ⇒ summed: diffusion-weighted signal of each voxel for each direction |
| $B_{gm}$ | `design_matrix(g)` | (G, 6) | rows $(g_x^2, g_y^2, g_z^2, 2g_xg_y, 2g_xg_z, 2g_yg_z)$ so that $\hat g_i D_{ij}\hat g_j = B_{gm}d_m$ |
| $d_{vm}$ | `torch.linalg.lstsq(-b * B, log(S / S0).T)` | (V, 6) | least-squares fit of the six unique tensor components, all voxels at once |
| $\lambda_{va},\ e_{vai}$ | `torch.linalg.eigh(D)` | (V, 3), (V, 3, 3) | eigenvalues and eigenvectors; the eigenvector of the largest eigenvalue points along the fibre |
| $MD_v,\ FA_v$ | `invariants(D)` | (V,) | mean diffusivity (trace/3) and fractional anisotropy (0 isotropic, 1 a single direction) |

Diffusivities are in $10^{-3}\,\mathrm{mm^2/s}$ and $b$ in $\mathrm{s/mm^2}$: roughly $\lambda_\parallel = 1.7$, $\lambda_\perp = 0.3$ in a bundle (FA ≈ 0.8), 3.0 for free water (CSF) and 0.8 for grey matter. These are approximate values for teaching, not for clinical use. The noise is Rician (the magnitude of a complex signal), so noisy fits overestimate FA in isotropic regions. Where two bundles cross, one tensor can only represent their average, a flat (planar) tensor with lower FA, and a tract entering it may stop or take the wrong turn into the other bundle: the classic limitation of DTI that multi-fibre models address. Each ellipse shows the in-plane shape of the fitted tensor. Streamlines start from seeds and follow the principal eigenvector of the bilinearly interpolated tensor field with the midpoint (RK2) rule until FA drops below 0.2. Try more noise, fewer gradient directions or a smaller $b$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/medical/diffusion_tensor_imaging/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n, b = params["size"], params["b"] * 1e-3  # b in s/mm² → per (10⁻³ mm²/s) units of D
    g = tensors.gradient_directions(params["n_dirs"], dev)  # (G, 3)
    D_true, fibre, label = tensors.phantom(n, dev)  # (V, 3, 3), (V, 3), (V,)
    B = tensors.design_matrix(g)  # (G, 6)

    S0, S = solve.measure(D_true, g, b, params["noise"])
    D_fit, lam, e1, md, fa = solve.fit(S0, S, B, b)

    names = ["grey matter", "arc bundle", "vertical bundle", "crossing", "CSF"]
    print("region            MD (1e-3 mm²/s)   FA     |e1 · fibre|")
    for k, name in enumerate(names):
        sel = label == k
        align = (e1[sel] * fibre[sel]).sum(-1).abs().mean() if k in (1, 2) else float("nan")
        print(f"{name:16s} {md[sel].mean():10.3f}      {fa[sel].mean():6.3f}   {align:8.3f}")
    print(f"max |D_fit - D_true| = {(D_fit - D_true).abs().max():.2e} (noise σ = {params['noise']})")

    lines = solve.track(D_fit.reshape(n, n, 3, 3), tensors.seeds(n, dev))
    print(f"{lines.shape[1]} streamlines, longest {int((~lines[..., 0].isnan()).sum(0).max())} RK2 steps")
    return plot.animate(D_fit, e1, fa, lines, n, out_dir / "diffusion_tensor_imaging.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/medical/diffusion_tensor_imaging/equation.py
r"""Physics → tensor translation.

Stejskal–Tanner: a diffusion-weighting gradient along the unit vector \hat g with strength b attenuates the
signal of voxel v by the diffusion coefficient along \hat g, a quadratic form of the diffusion tensor:

    S_{vg} = S_{0v} \exp(-b\, \hat g_{gi} D_{vij} \hat g_{gj})          (i, j summed; batched over v and g)

Taking logs makes it linear in the 6 unique components d_{vm} of the symmetric tensor:

    \ln(S_{vg}/S_{0v}) = -b\, B_{gm} d_{vm}        (m summed)  →  least squares for every voxel at once

The eigen-decomposition D_{vij} = \sum_a \lambda_{va} e_{vai} e_{vaj} gives the fibre direction (largest λ) and

    MD = \lambda_a / 3 = D_{ii} / 3,      FA = \sqrt{3/2}\, \lVert \lambda - MD \rVert / \lVert \lambda \rVert
"""

import math

import torch


def signal(D: torch.Tensor, g: torch.Tensor, b: float, S0: float = 1.0) -> torch.Tensor:
    r"""S_{vg} = S_0 \exp(-b \hat g_{gi} D_{vij} \hat g_{gj}).   D: (V, 3, 3), g: (G, 3) → (V, G)"""
    adc = torch.einsum("gi,vij,gj->vg", g, D, g)  # i, j summed: apparent diffusion coefficient along each ĝ
    return S0 * torch.exp(-b * adc)


def fit_tensor(S: torch.Tensor, S0: torch.Tensor, B: torch.Tensor, b: float) -> torch.Tensor:
    r"""Least squares for -b B_{gm} d_{vm} = \ln(S_{vg}/S_{0v}), all voxels in one call.   → D (V, 3, 3)"""
    y = torch.log(S.clamp(min=1e-6) / S0.clamp(min=1e-6)[:, None])  # (V, G)
    d = torch.linalg.lstsq(-b * B, y.T).solution.T  # (G, 6) \ (G, V) → (6, V) → (V, 6)
    dxx, dyy, dzz, dxy, dxz, dyz = d.unbind(-1)
    rows = [torch.stack(r, -1) for r in ((dxx, dxy, dxz), (dxy, dyy, dyz), (dxz, dyz, dzz))]
    return torch.stack(rows, dim=-2)  # (V, 3, 3) symmetric


def invariants(D: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    r"""Eigenvalues λ (V, 3) ascending, principal eigenvector e_1 (V, 3), MD (V,) and FA (V,)."""
    lam, vec = torch.linalg.eigh(D)  # symmetric → real eigenvalues, orthonormal eigenvectors (columns)
    md = torch.einsum("vii->v", D) / 3  # trace / 3 = mean of the eigenvalues
    dev = lam - md[:, None]
    fa = math.sqrt(1.5) * dev.norm(dim=-1) / lam.norm(dim=-1).clamp(min=1e-12)
    return lam, vec[..., -1], md, fa


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/medical/diffusion_tensor_imaging/tensors.py
"""Build the concrete tensors: gradient directions, a 2D slice of voxels with fibre bundles, the true D_ij field."""

import math

import torch

# Approximate diffusivities in 10⁻³ mm²/s (= µm²/ms). For teaching only.
LAMBDA_PARALLEL = 1.7  # along an axon bundle
LAMBDA_PERP = 0.3  # across it
D_CSF = 3.0  # free water at body temperature
D_GREY = 0.8  # isotropic tissue between the bundles


def gradient_directions(n: int, device: torch.device) -> torch.Tensor:
    """ĝ_{gi}: n unit vectors spread evenly over a hemisphere (Fibonacci spiral). Shape (G, 3).

    ±ĝ give the same signal, so one hemisphere is enough.
    """
    k = torch.arange(n, dtype=torch.float64, device=device) + 0.5
    z = 1 - k / n  # from ~1 down to ~0: upper hemisphere
    phi = math.pi * (3 - math.sqrt(5)) * k  # golden-angle steps in azimuth
    r = torch.sqrt(1 - z**2)
    return torch.stack([r * torch.cos(phi), r * torch.sin(phi), z], dim=-1)


def fibre_tensor(e: torch.Tensor, lam_par: float, lam_perp: float) -> torch.Tensor:
    """D_{ij} = λ∥ e_i e_j + λ⊥ (δ_ij − e_i e_j) for unit fibre directions e (..., 3) → (..., 3, 3)."""
    eye = torch.eye(3, dtype=e.dtype, device=e.device)
    ee = torch.einsum("...i,...j->...ij", e, e)
    return lam_par * ee + lam_perp * (eye - ee)


def phantom(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """A slice of n × n voxels: an arched bundle, a vertical bundle crossing it, and a CSF-filled ventricle.

    Returns the true tensors D_{vij} (V, 3, 3) with V = n², the fibre direction of each voxel (V, 3; zero where
    there is no single fibre) and a label per voxel (V,): 0 grey matter, 1 arc, 2 vertical, 3 crossing, 4 CSF.
    Voxel v = row · n + column; the coordinates (x, y) = (column, row) with y pointing up.
    """
    c = torch.arange(n, dtype=torch.float64, device=device)
    y, x = torch.meshgrid(c, c, indexing="ij")
    s = n / 20  # sizes below are for a 20-voxel slice
    cx, cy = 9.5 * s, 0.0
    r = torch.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    arc = (r > 10 * s) & (r < 14 * s)
    vertical = (x > 13.5 * s) & (x < 16.5 * s)
    csf = ((x - cx) / (3.0 * s)) ** 2 + ((y - 4 * s) / (2.2 * s)) ** 2 < 1

    zero = torch.zeros_like(x)
    e_arc = torch.stack([-(y - cy), x - cx, zero], dim=-1) / r.clamp(min=1e-9)[..., None]  # tangent to circles
    e_up = torch.stack([zero, torch.ones_like(x), zero], dim=-1)

    D = D_GREY * torch.eye(3, dtype=torch.float64, device=device).expand(n, n, 3, 3).clone()
    D_arc = fibre_tensor(e_arc, LAMBDA_PARALLEL, LAMBDA_PERP)
    D_up = fibre_tensor(e_up, LAMBDA_PARALLEL, LAMBDA_PERP)
    D = torch.where(arc[..., None, None], D_arc, D)
    D = torch.where(vertical[..., None, None], D_up, D)
    D = torch.where((arc & vertical)[..., None, None], 0.5 * (D_arc + D_up), D)  # crossing: average of the two
    D = torch.where(csf[..., None, None], D_CSF * torch.eye(3, dtype=torch.float64, device=device), D)

    label = torch.zeros(n, n, dtype=torch.long, device=device)
    label[arc], label[vertical], label[arc & vertical], label[csf] = 1, 2, 3, 4
    fibre = torch.where((label == 1)[..., None], e_arc, torch.zeros_like(e_arc))
    fibre = torch.where((label == 2)[..., None], e_up, fibre)
    return D.reshape(-1, 3, 3), fibre.reshape(-1, 3), label.reshape(-1)


def design_matrix(g: torch.Tensor) -> torch.Tensor:
    """B_{gm}: ĝ_i D_ij ĝ_j = B_{gm} d_m for the 6 unique components d = (Dxx, Dyy, Dzz, Dxy, Dxz, Dyz). (G, 6)"""
    gx, gy, gz = g.unbind(-1)
    return torch.stack([gx * gx, gy * gy, gz * gz, 2 * gx * gy, 2 * gx * gz, 2 * gy * gz], dim=-1)


def seeds(n: int, device: torch.device) -> torch.Tensor:
    """Tractography start points (P, 2) in voxel coordinates: across the arc's top and the vertical bundle's foot."""
    s = n / 20
    across_arc = torch.stack([torch.full((6,), 9.5 * s), torch.linspace(10.4 * s, 13.6 * s, 6)], dim=-1)
    foot = torch.stack([torch.linspace(13.9 * s, 16.1 * s, 4), torch.full((4,), 1.0 * s)], dim=-1)
    return torch.cat([across_arc, foot]).to(device=device, dtype=torch.float64)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/medical/diffusion_tensor_imaging/solve.py
"""Measure noisy DW signals, fit a tensor per voxel, and trace fibres along the principal eigenvector with RK2."""

import torch

from problems._common.progress import steps

from .equation import fit_tensor, invariants, signal


def measure(
    D: torch.Tensor, g: torch.Tensor, b: float, noise: float, seed: int = 0
) -> tuple[torch.Tensor, torch.Tensor]:
    """b = 0 image S0 (V,) and diffusion-weighted images S (V, G) with Rician noise of std `noise` (S0 = 1).

    Rician noise: the scanner measures the magnitude of a complex signal, |S + σ(n₁ + i n₂)|.
    """
    gen = torch.Generator(device=D.device).manual_seed(seed)
    clean = torch.cat([torch.ones(D.shape[0], 1, dtype=D.dtype, device=D.device), signal(D, g, b)], dim=1)
    n1 = torch.randn(clean.shape, generator=gen, dtype=D.dtype, device=D.device)
    n2 = torch.randn(clean.shape, generator=gen, dtype=D.dtype, device=D.device)
    noisy = torch.sqrt((clean + noise * n1) ** 2 + (noise * n2) ** 2)
    return noisy[:, 0], noisy[:, 1:]


def fit(S0: torch.Tensor, S: torch.Tensor, B: torch.Tensor, b: float):
    """Fitted tensors (V, 3, 3) and their invariants: eigenvalues, principal direction, MD, FA."""
    D_fit = fit_tensor(S, S0, B, b)
    return (D_fit, *invariants(D_fit))


def _interpolate(field: torch.Tensor, pos: torch.Tensor) -> torch.Tensor:
    """Bilinear interpolation of a voxel field (n, n, ...) at positions pos (P, 2) = (x, y). → (P, ...)"""
    n = field.shape[0]
    x = pos[:, 0].clamp(0, n - 1)
    y = pos[:, 1].clamp(0, n - 1)
    x0 = x.floor().long().clamp(max=n - 2)
    y0 = y.floor().long().clamp(max=n - 2)
    fx, fy = x - x0, y - y0
    shape = (-1,) + (1,) * (field.dim() - 2)
    fx, fy = fx.view(shape), fy.view(shape)
    return (
        field[y0, x0] * (1 - fx) * (1 - fy)
        + field[y0, x0 + 1] * fx * (1 - fy)
        + field[y0 + 1, x0] * (1 - fx) * fy
        + field[y0 + 1, x0 + 1] * fx * fy
    )


def _direction(D_grid: torch.Tensor, pos: torch.Tensor, previous: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """In-plane principal direction (P, 2) at pos, flipped to agree with `previous`, and the local FA (P,)."""
    D = _interpolate(D_grid, pos)  # (P, 3, 3)
    _, e1, _, fa = invariants(D)
    d = e1[:, :2] / e1[:, :2].norm(dim=-1, keepdim=True).clamp(min=1e-12)
    sign = torch.sign(torch.einsum("pi,pi->p", d, previous))  # eigenvectors have no sign: keep going forward
    return d * torch.where(sign == 0, 1.0, sign)[:, None], fa


def track(
    D_grid: torch.Tensor, start: torch.Tensor, h: float = 0.3, n_steps: int = 90, fa_stop: float = 0.2
) -> torch.Tensor:
    """Streamlines (S+1, 2P, 2) from every seed in both directions; NaN after a line stops.

    Midpoint rule (RK2):  k₁ = e(x),  k₂ = e(x + h/2 k₁),  x ← x + h k₂.  A line stops where FA < fa_stop
    (CSF, grey matter, or a crossing) or when it leaves the slice.
    """
    n = D_grid.shape[0]
    first, _ = _direction(D_grid, start, torch.ones_like(start))
    pos = torch.cat([start, start])  # (2P, 2)
    heading = torch.cat([first, -first])  # forward and backward
    alive = torch.ones(len(pos), dtype=torch.bool, device=pos.device)
    path = [pos.clone()]
    for _ in steps(n_steps, "tracking steps"):
        k1, fa = _direction(D_grid, pos, heading)
        k2, _ = _direction(D_grid, pos + 0.5 * h * k1, k1)
        new = pos + h * k2
        inside = (new >= 0).all(-1) & (new <= n - 1).all(-1)
        alive = alive & inside & (fa >= fa_stop)
        pos = torch.where(alive[:, None], new, pos)
        heading = torch.where(alive[:, None], k2, heading)
        path.append(torch.where(alive[:, None], pos, torch.full_like(pos, float("nan"))))
    return torch.stack(path)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/medical/diffusion_tensor_imaging/plot.py
"""Diffusion ellipses on the FA map (left) and streamlines growing over the colour-FA map (right)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import EllipseCollection

from problems._common.plotting import frame_indices, save_gif


def animate(D: torch.Tensor, e1: torch.Tensor, fa: torch.Tensor, lines: torch.Tensor, n: int, path: Path) -> Path:
    fa_map = fa.reshape(n, n).clamp(0, 1).cpu().numpy()
    colour = (fa[:, None] * e1.abs()).clamp(0, 1).reshape(n, n, 3).cpu().numpy()  # red = x, green = y, blue = z
    lam2, vec2 = torch.linalg.eigh(D[:, :2, :2])  # in-plane 2×2 block → ellipse axes
    widths = torch.full_like(lam2[:, 1], 0.9).cpu().numpy()  # every glyph spans most of its voxel ...
    heights = (0.9 * lam2[:, 0] / lam2[:, 1].clamp(min=1e-9)).cpu().numpy()  # ... squashed by λ_small/λ_large
    angles = torch.rad2deg(torch.atan2(vec2[:, 1, 1], vec2[:, 0, 1])).cpu().numpy()
    yy, xx = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    centres = np.stack([xx.ravel(), yy.ravel()], axis=1)
    tracks = lines.cpu().numpy()  # (S, L, 2)

    fig, (ax_g, ax_t) = plt.subplots(1, 2, figsize=(7.6, 3.9))
    ax_g.imshow(fa_map, cmap="gray", vmin=0, vmax=1, origin="lower")
    ax_g.add_collection(
        EllipseCollection(
            widths,
            heights,
            angles,
            units="xy",
            offsets=centres,
            offset_transform=ax_g.transData,
            facecolors=colour.reshape(-1, 3) * 0.7 + 0.3,
            edgecolors="#1f2937",
            linewidths=0.3,
        )
    )
    ax_g.set_title("diffusion ellipses on the FA map", fontsize=9)
    ax_t.imshow(colour, origin="lower")
    ax_t.set_title("colour FA (red x, green y) + tractography", fontsize=9)
    curves = [ax_t.plot([], [], color="white", lw=1.4)[0] for _ in range(tracks.shape[1])]
    heads = ax_t.scatter(np.zeros(tracks.shape[1]), np.zeros(tracks.shape[1]), s=8, c="#facc15", zorder=3)
    for ax in (ax_g, ax_t):
        ax.set_xlim(-0.5, n - 0.5)
        ax.set_ylim(-0.5, n - 0.5)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)

    def update(step: int):
        for line, curve in zip(np.transpose(tracks[: step + 1], (1, 0, 2)), curves):
            curve.set_data(line[:, 0], line[:, 1])
        tips = np.array([line[~np.isnan(line[:, 0])][-1] for line in np.transpose(tracks[: step + 1], (1, 0, 2))])
        heads.set_offsets(tips)

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(tracks), 60), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'b': 1000.0,
    'n_dirs': 30,
    'noise': 0.02,
    'size': 20
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.medical.diffusion_tensor_imaging")]:
    del sys.modules[name]
main = importlib.import_module("problems.medical.diffusion_tensor_imaging.main")
problem_dir = Path("problems/medical/diffusion_tensor_imaging")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
